# **Scraping Planos Vivo**

_Página de planos do provedor: https://vivo.com.br/para-voce/produtos-e-servicos/para-casa/internet_

## **1. Extração**
Faz extração e retorna um um arquivo vivo.json

In [ ]:
from playwright.async_api import async_playwright
import asyncio
import json
import os

async def run():
    async with async_playwright() as pw:
        navegador = await pw.chromium.launch(headless=False)
        contexto = await navegador.new_context()
        pagina = await contexto.new_page()

        await pagina.goto("https://vivo.com.br/para-voce/produtos-e-servicos/para-casa/internet")

        await pagina.get_by_role("link", name="Trocar localização").click()

        campo = pagina.get_by_role("textbox", name="Preencha com o nome da sua")
        await campo.fill("belo horizonte")

        await pagina.get_by_role("link", name="Belo Horizonte (MG)").click()
        await pagina.get_by_role("button", name="Confirmar").click()

        await pagina.wait_for_load_state("networkidle")

        # ---- CAPTURA DO HTML ----
        html = await pagina.content()

        path_html = "../../data/scraped/vivo.html"
        os.makedirs(os.path.dirname(path_html), exist_ok=True)

        with open(path_html, "w", encoding="utf-8") as f:
            f.write(html)

        # ---- SEGUE FLUXO NORMAL ----
        await pagina.get_by_role("button", name="Consultar Fibra").first.click()

        response = await contexto.request.get(
            "https://loja.vivo.com.br/bff/unicommerceTacticalB2CBff/v1/catalogs"
        )

        data = await response.json()

        path_json = "../../data/scraped/vivo.json"
        os.makedirs(os.path.dirname(path_json), exist_ok=True)

        with open(path_json, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        await contexto.close()
        await navegador.close()

await run()

## **2 Tratamento**

### **2.1 Lista de códigos de produtos/promoções exibidas na página**

In [7]:
from bs4 import BeautifulSoup
from urllib.parse import parse_qs
import html

soup = BeautifulSoup(open("../../data/scraped/vivo.html", encoding="utf-8"), "html.parser")

planos_na_pagina = list({
    f'{q.get("productsIds", [""])[0]}|@|{q.get("promotion", [""])[0]}'
    for a in soup.find_all("a", href=True)
    if "vivofibra" in a["href"]
    for q in [parse_qs(html.unescape(a["href"]).split("?")[1])]
})

print(planos_na_pagina)

['2|@|Pro_160', '106|@|GAMER', '235,266|@|600', '105|@|', '127|@|', '235,328|@|Fibra100', '235,267|@|600', '104,327|@|', '317,328|@|Fibra100', '117,327|@|', '105|@|GAMER', '235,265|@|600', '235|@|600', '317,329|@|Fibra100', '235,264|@|600', '117|@|', '122|@|', '106|@|', '235,329|@|Fibra100']


### **2.2 Filtro: apenas planos na página**

In [8]:
import pandas as pd

df = pd.read_json("../../data/scraped/vivo.json")

df = df[
    (
        df["codigoDaOferta"].fillna("").astype(str)
        + "|@|"
        + df["codigoPromocional"].fillna("").astype(str)
    ).isin(planos_na_pagina)
]

df.head()

,ofertaMacro,ativa,nomeComercial,codigoDaOferta,codigoPromocional,steps,textoOferta,vivoTotal,vivoTotalV1V2,TV Solo?,...,Oferta,Oferta estendida,Parceiro,Token carrinho abandonado homologação,Token pedido fechado homologação,Token carrinho abandonado produção,Token pedido fechado produção,Token Topaz homologação,Token Topaz produção,isRecorderEnable
0,Fibra solo,Sim,VIVO FIBRA 600 MEGA,235,600,4A,Pague 500 Mega e leve 600 Mega! Oferta exclusi...,Não,Não,Não,...,NaN,NaN,NaN,WxN9Dn2k%2BD0VDikuONhHTPM4oVHbK3HRpfu64bcDgHk,%2B%2F%2FD5Rh%2F3YH24CjFjc89L2X3lDDPCPN6i%2F9l...,F7jUlBR%2B%2FN22HP0QdnDuJubYzHXDsthq9xq5MVNFJUs,s8VdIt2CU0nFlrm7uBj%2Ba9tiBWCwi7RCMt2ONZYBXYo,7eXbzC9mPxv3W0TnTyLwV2l7f0BwDUQbBQZkbWndCRc,3rZQbrrKGnhAHIt672eqMmlmjoqSBbBMqHD0BRPKMbE,NaN
5,Fibra solo,Sim,VIVO FIBRA 700 MEGA,105,None,4A,NaN,Não,Não,Não,...,NaN,NaN,NaN,WxN9Dn2k%2BD0VDikuONhHTPM4oVHbK3HRpfu64bcDgHk,eGlLgtsWYhwoP%2BarehF6zquFqxVzrzjK5Jrqf5baC4E,F7jUlBR%2B%2FN22HP0QdnDuJubYzHXDsthq9xq5MVNFJUs,FohHYGq9BOouIFHRYDZ%2FiJga8dtgOuRKAUG1QZ0U14s,7eXbzC9mPxv3W0TnTyLwV2l7f0BwDUQbBQZkbWndCRc,3rZQbrrKGnhAHIt672eqMmlmjoqSBbBMqHD0BRPKMbE,NaN
6,Fibra solo,Sim,VIVO FIBRA 1 GIGA,106,None,4A,NaN,Não,Não,Não,...,NaN,NaN,NaN,WxN9Dn2k%2BD0VDikuONhHTPM4oVHbK3HRpfu64bcDgHk,eGlLgtsWYhwoP%2BarehF6zquFqxVzrzjK5Jrqf5baC4E,F7jUlBR%2B%2FN22HP0QdnDuJubYzHXDsthq9xq5MVNFJUs,FohHYGq9BOouIFHRYDZ%2FiJga8dtgOuRKAUG1QZ0U14s,7eXbzC9mPxv3W0TnTyLwV2l7f0BwDUQbBQZkbWndCRc,3rZQbrrKGnhAHIt672eqMmlmjoqSBbBMqHD0BRPKMbE,NaN
9,Fibra Gamer,Sim,VIVO FIBRA 700 MEGA + GAMER,105,GAMER,4A,NaN,Não,Não,Não,...,NaN,NaN,NaN,WxN9Dn2k%2BD0VDikuONhHTPM4oVHbK3HRpfu64bcDgHk,q%2F%2FaSXdgjV81etO%2FDjeenCQbOLA9S4uP5PC%2Ba5...,F7jUlBR%2B%2FN22HP0QdnDuJubYzHXDsthq9xq5MVNFJUs,cXE3bJELr2Do9ibAjAhh1%2Fi5R%2FN3URM7e%2BvzLz4pkxg,7eXbzC9mPxv3W0TnTyLwV2l7f0BwDUQbBQZkbWndCRc,3rZQbrrKGnhAHIt672eqMmlmjoqSBbBMqHD0BRPKMbE,NaN
10,Fibra Gamer,Sim,VIVO FIBRA 1 GIGA + GAMER,106,GAMER,4A,NaN,Não,Não,Não,...,NaN,NaN,NaN,WxN9Dn2k%2BD0VDikuONhHTPM4oVHbK3HRpfu64bcDgHk,q%2F%2FaSXdgjV81etO%2FDjeenCQbOLA9S4uP5PC%2Ba5...,F7jUlBR%2B%2FN22HP0QdnDuJubYzHXDsthq9xq5MVNFJUs,cXE3bJELr2Do9ibAjAhh1%2Fi5R%2FN3URM7e%2BvzLz4pkxg,7eXbzC9mPxv3W0TnTyLwV2l7f0BwDUQbBQZkbWndCRc,3rZQbrrKGnhAHIt672eqMmlmjoqSBbBMqHD0BRPKMbE,NaN


### **2.3 Tratamento das colunas de interesse**

In [ ]:
from datetime import date
import re

# Data e provedor
df["DATA"] = date.today().strftime("%Y-%m-%d")
df["ISP"] = "Vivo"

# Velocidade
df["VELOCIDADE"] = df["textoFibra"].str.extract(r"(\d+)").astype(float)
df.loc[
    df["textoFibra"].str.contains(r"giga|gb|g\b", case=False, na=False, regex=True),
    "VELOCIDADE"
] *= 1000

# SVA: cria uma lista as colunas que iniciam com "cortesiaFibra" ou "sva" ou "textoTv"
# depois concatena essas colunas
cols_sva = [c for c in df if c.startswith(("cortesiaFibra", "sva", "textoTv"))]

df["SVAs"] = df[cols_sva].apply(
    lambda row: ", ".join(row.dropna().astype(str)),
    axis=1
)

# Preço
df["PRECO"] = (
    df["total"]
    .str.replace(r"[^\d,]", "", regex=True)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

#Seleção de colunas
df_tratado = df[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

,DATA,ISP,VELOCIDADE,SVAs,PRECO
0,2026-04-19,Vivo,600.0,Amazon Prime,100.0
5,2026-04-19,Vivo,700.0,Amazon Prime,150.0
6,2026-04-19,Vivo,1000.0,Amazon Prime,300.0
9,2026-04-19,Vivo,700.0,"Amazon Prime, Ponto Cabeado, Exit Lag",170.0
10,2026-04-19,Vivo,1000.0,"Amazon Prime, Ponto Cabeado, Exit Lag",320.0
11,2026-04-19,Vivo,500.0,"Amazon Prime, TV Avançado",270.0
13,2026-04-19,Vivo,700.0,"Amazon Prime, TV Avançado",320.0
14,2026-04-19,Vivo,1000.0,"Amazon Prime, TV Avançado",470.0
31,2026-04-19,Vivo,500.0,"Amazon Prime, Netflix Padrão",144.9
53,2026-04-19,Vivo,600.0,"Amazon Prime, Serviço de automonitoramento",120.0


## **3. Carga**

In [10]:
# Salvando
nome_arq = df_tratado["DATA"].max() + "-VIVO.csv"

#pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
#df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

df_tratado.to_csv(f"../../data/flat/{nome_arq}", index=False)
print(f"🟢 {nome_arq} salvo!")

🟢 2026-04-19-VIVO.csv salvo!
